## Investigating the Second Pass

This brief notebook downloads tagged Loci from ANTARES (our Targets), classifies them using our _Second Pass_ classifier (which still needs further development), and then offers the ability to visualise the Candidates which pass the _Second Pass_ using visualisation code from Shenming. The _Second Pass_ classifier being used here comes from the `lantern_second_pass_classifier.ipynb` notebook, and can be updated as needed. A trained version of this model is available in the shared Google Drive; it uses the data and setup currently in the notebook. Changes to this model should be tracked and saved!

Note that the query implementation is still finicky--as of now, it seems to query in reverse time order (meaning, it queries the most recently tagged Targets first). A future iteration of this should make sure to implement a way to avoid repeating searches, so that we don't have to query the full tagged list every time. 

Note also that we do not currently implement any clustering, as all test runs have returned Loci which are not able to be clustered within the 3 arcesec that we are interested in. 

In [ ]:
from antares_client.search import search
from antares_client.search import get_by_id, get_by_ztf_object_id, get_by_lsst_dia_object_id


import requests
from io import BytesIO
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from astropy.io import fits
from astropy.visualization import make_lupton_rgb

import json
from json import JSONDecodeError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Functions

In [ ]:
def process_locus(locus):
    
    print(locus.locus_id)

    ra = locus.ra
    dec = locus.dec

    return ra, dec


def _build_url(ra, dec, fmt="jpg", layer="ls-dr10", pixscale=0.263,
               size=50, bands=None):
    params = {"ra": ra, "dec": dec, "layer": layer,
              "pixscale": pixscale, "size": size}
    if bands is not None:
        params["bands"] = bands
    qs = "&".join(f"{k}={v}" for k, v in params.items())
    return f"{BASE_URL}.{fmt}?{qs}"


def show_cutout(ra, dec, size=50, layer="ls-dr10", pixscale=0.263,
                title=None, figsize=(3, 3)):
    """Fetch a JPG cutout from Legacy Survey and display it inline."""
    url = _build_url(ra, dec, "jpg", layer, pixscale, size)
    r = requests.get(url, timeout=30); r.raise_for_status()
    img = PILImage.open(BytesIO(r.content))

    fov = size * pixscale / 60.0
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title or f"RA={ra:.4f}, Dec={dec:+.4f}\n({fov:.2f}′|{size}pix, {layer})")
    plt.xticks([]); plt.yticks([])
    plt.show()
    return url


def show_fits_rgb(ra, dec, size=50, layer="ls-dr10", pixscale=0.263,
                  bands="grz", stretch=0.05, Q=5, minimum=-0.05,
                  title=None, figsize=(3, 3), return_data=False):
    """
    Fetch a grz FITS cutout and render it as a Lupton RGB image.
    Channel mapping: z → R, r → G, g → B.
    """
    url = _build_url(ra, dec, "fits", layer, pixscale, size, bands=bands)
    r = requests.get(url, timeout=60); r.raise_for_status()
    hdul = fits.open(BytesIO(r.content))

    cube = hdul[0].data
    if cube.ndim != 3 or cube.shape[0] != len(bands):
        raise ValueError(f"Unexpected FITS shape {cube.shape} for bands={bands}")

    imgs = {b: cube[i] for i, b in enumerate(bands)}
    rgb = make_lupton_rgb(imgs["z"], imgs["r"], imgs["g"],
                          minimum=minimum, stretch=stretch, Q=Q)

    fov = size * pixscale / 60.0
    plt.figure(figsize=figsize)
    plt.imshow(rgb, origin="lower")
    plt.title(title or f"RA={ra:.4f}, Dec={dec:+.4f}\n"
                       f"({fov:.2f}′|{size}pix, {layer}, Lupton g/r/z)")
    plt.xticks([]); plt.yticks([])
    plt.show()

    #return (rgb, hdul, url) if return_data else url

def scan_lupton(ra, dec, size=256, layer="ls-dr10", pixscale=0.262,
                stretches=(0.5, 0.1, 0.05, 0.01),
                Qs=(5, 10, 20), minimum=-0.05):
    """Grid of Lupton RGB renderings over (stretch, Q)."""
    url = _build_url(ra, dec, "fits", layer, pixscale, size, bands="grz")
    r = requests.get(url, timeout=60); r.raise_for_status()
    cube = fits.open(BytesIO(r.content))[0].data
    g, r_, z = cube[0], cube[1], cube[2]

    nrows, ncols = len(Qs), len(stretches)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = axes.reshape(nrows, ncols)
    for i, Q in enumerate(Qs):
        for j, s in enumerate(stretches):
            rgb = make_lupton_rgb(z, r_, g, minimum=minimum, stretch=s, Q=Q)
            ax = axes[i, j]
            ax.imshow(rgb, origin="lower")
            ax.set_title(f"stretch={s}, Q={Q}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

def show_by_id(locus_id):
    locus = get_by_id(locus_id)

    ra, dec = process_locus(locus)
    show_cutout(ra, dec, size=50)
    show_fits_rgb(ra, dec, size=50)

def show_by_dia_obj_id(lsst_dia_object_id):
    locus = get_by_lsst_dia_object_id(lsst_dia_object_id)

    ra, dec = process_locus(locus)
    show_cutout(ra, dec, size=50)
    show_fits_rgb(ra, dec, size=50)

def get_prop(alert, name):
    if alert.properties.get(name) is not None:
        val = alert.properties[name]
    else:
        val = np.nan
    return val

In [ ]:
def normalize_bands_and_flags(df):
            """Normalizes band strings and converts flag columns. Returns a copy."""
            df = df.copy()
            if 'band' in df.columns:
                # Handle both FilterLabel(band="g", ...) and plain 'g' formats
                extracted = df['band'].astype(str).str.extract(r"band=['\"]([ugrizy])['\"]", expand=False)
                # Fall back to bare letter for rows already in plain format
                plain = df['band'].astype(str).str.extract(r'^([ugrizy])$', expand=False)
                df['band'] = extracted.fillna(plain)
                # XGBoost requires categorical columns to be the 'category' dtype
                df['band'] = df['band'].astype('category')
            flag_cols = [c for c in df.columns if 'flag' in c.lower() or c in ['isDipole']]
            for col in flag_cols:
                df[col] = df[col].astype(bool)
            return df


def _add_engineered_features(alert):
        """
        Calculate engineered features from a single alert.
    
        Returns
        -------
        dict
            Dictionary of engineered features
        """
        import numpy as np
        
        def get_prop(name):
            if alert.properties.get(name) is not None:
                val = alert.properties[name]
            else:
                val = np.nan
            return val

        # Calculate moment extension
        psf_trace = get_prop('lsst_diaSource_ixxPSF') + get_prop('lsst_diaSource_iyyPSF')
        src_trace = get_prop('lsst_diaSource_ixx') + get_prop('lsst_diaSource_iyy')
        moment_ext = src_trace / psf_trace if psf_trace != 0 else np.nan

         # Calculate ellipticity extension
        ixx = get_prop('lsst_diaSource_ixx')
        iyy = get_prop('lsst_diaSource_iyy')
        ixy = get_prop('lsst_diaSource_ixy')
        ixxPSF = get_prop('lsst_diaSource_ixxPSF')
        iyyPSF = get_prop('lsst_diaSource_iyyPSF')
        ixyPSF = get_prop('lsst_diaSource_ixyPSF')

        if src_trace != 0:
            src_ellip = np.sqrt((ixx - iyy)**2 + 4 * ixy**2) / src_trace
        else:
            src_ellip = np.nan
        
        if psf_trace != 0:
            psf_ellip = np.sqrt((ixxPSF - iyyPSF)**2 + 4 * ixyPSF**2) / psf_trace
        else:
            psf_ellip = np.nan
        
        ellip_ext = src_ellip - psf_ellip

        # Calculate flux extension
        psfFlux = get_prop('lsst_diaSource_psfFlux')
        apFlux = get_prop('lsst_diaSource_apFlux')
        flux_ext = apFlux / psfFlux if psfFlux != 0 else np.nan
        
        # Calculate template flux ratio
        scienceFlux = get_prop('lsst_diaSource_scienceFlux')
        template_flux = scienceFlux - psfFlux
        temp_sci_flux_ratio = template_flux / scienceFlux if scienceFlux != 0 else np.nan
        
        # Calculate PSF FWHM
        psf_det = ixxPSF * iyyPSF - ixyPSF**2
        psf_fwhm = np.abs(psf_det)**(1/4) * 2.35482 * 0.2 if psf_det != 0 else np.nan
        
        # Calculate x-y error
        xErr = get_prop('lsst_diaSource_xErr')
        yErr = get_prop('lsst_diaSource_yErr')
        x_y_err = np.sqrt(xErr**2 + yErr**2)
        
        # Return as dictionary
        return {
            'moment_ext': moment_ext,
            'ellip_ext': ellip_ext,
            'flux_ext': flux_ext,
            'template_flux': template_flux,
            'temp_sci_flux_ratio': temp_sci_flux_ratio,
            'psf_fwhm': psf_fwhm,
            'x_y_err': x_y_err
        }

In [ ]:
def calculate_stats(df):
    """
    Calculate centroid instability and band-specific flux statistics for each unique object.
    
    Parameters
    ----------
    df : DataFrame
        DataFrame where each row is a detection, with columns:
        x, y, xErr, yErr, x_y_err, band, apFlux, template_flux, flux_ext, 
        moment_ext, extendedness, and lens_id
    
    Returns
    -------
    DataFrame
        One row per unique object with centroid statistics and per-band flux measurements
    """
    # Group by object identifier
    # grouped = df.groupby('lsst_dia_obj_id')
    grouped = df.groupby('locus_id')
    
    summary = grouped.agg(
        n_detections=('x', 'count'),
        x_std=('x', 'std'),
        y_std=('y', 'std'),
        mean_x_y_err=('x_y_err', 'mean'),
        median_x_y_err=('x_y_err', 'median'),

        # Global (not band-specific) statistics
        moment_ext_mean=('moment_ext', 'mean'),
        moment_ext_std=('moment_ext', 'std'),
        extendedness_mean=('extendedness', 'mean'),
        extendedness_std=('extendedness', 'std'),
        dipoleLength_mean = ('dipoleLength', 'mean'),
        dipoleLength_std = ('dipoleLength', 'std'),

        #Constant characteristics
        # category=('category', 'first'),
        # y_true=('y_true', 'first'),
        ra=('ra', 'first'),
        dec=('dec', 'first'),
        n_tagged_alerts=('n_alerts', 'first'),
        n_lsst_alerts=('alert_id', 'max')

    ).reset_index()
    
    # Calculate combined centroid standard deviation
    summary['centroid_std'] = np.sqrt(summary['x_std']**2 + summary['y_std']**2)
    
    # Normalize by measurement error (instability metric)
    summary['centroid_instability'] = np.where(
        summary['median_x_y_err'] > 0,
        summary['centroid_std'] / summary['median_x_y_err'],
        0.0
    )
    
    # Handle single-detection objects (can't calculate std)
    summary['centroid_std'] = summary['centroid_std'].fillna(0.0)
    summary['centroid_instability'] = summary['centroid_instability'].fillna(0.0)
    
    # Calculate per-band statistics
    bands = ['u', 'g', 'r', 'i', 'z', 'y']
    
    for band in bands:
        # Filter to this band
        band_data = df[df['band'] == band].groupby('locus_id').agg(
            **{
                f'apFlux_mean_{band}': ('apFlux', 'mean'),
                f'apFlux_std_{band}': ('apFlux', 'std'),
                f'template_flux_mean_{band}': ('template_flux', 'mean'),
                f'flux_ext_mean_{band}': ('flux_ext', 'mean'),
                f'flux_ext_std_{band}': ('flux_ext', 'std'),
            }
        )
        
        # Merge with summary
        summary = summary.merge(band_data, left_on='locus_id', right_index=True, how='left')
    
    # Fill NaN for bands with no detections
    for band in bands:
        summary[f'apFlux_mean_{band}'] = summary[f'apFlux_mean_{band}'].fillna(np.nan)
        summary[f'apFlux_std_{band}'] = summary[f'apFlux_std_{band}'].fillna(np.nan)
        summary[f'template_flux_mean_{band}'] = summary[f'template_flux_mean_{band}'].fillna(np.nan)
        summary[f'flux_ext_mean_{band}'] = summary[f'flux_ext_mean_{band}'].fillna(np.nan)
        summary[f'flux_ext_std_{band}'] = summary[f'flux_ext_std_{band}'].fillna(np.nan)
    
    return summary

In [ ]:
import pickle

with open('second_pass_first_test.pkl', 'rb') as file:
    lantern = pickle.load(file)

In [ ]:
lantern

### Implementation

In [ ]:
QUERY = {
    "query": {
        "bool": {
            "filter": {
                "terms": {
                    "tags": [
                        "lantern_xgboost_t2.0.7_c0.95"
                    ]
                }
            }
        }
    }
}


BASE_URL = "https://www.legacysurvey.org/viewer/cutout"

In [ ]:
#note: 100 loci takes about 90 seconds

res = {}
lim = 10_000 #returns this number of loci

for ind, locus in enumerate(search(QUERY)):
    if ind==lim: break
    max_score = locus.properties['lantern_xgboost_t2.0.7_c0.95_max_score']
    if max_score > 0.9691:
        locus_id = locus.locus_id
        # obj_id = locus.properties['survey']['lsst']['dia_object_id'][0]
        n_alerts = locus.properties['lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts']
        # ra = locus.ra
        # dec = locus.dec
        alert_num = 0

        try:
            for alert in locus.alerts:
                if 'lsst_diaSource_band' in alert.properties:
                    # Extract direct alert properties
                    band = get_prop(alert, 'lsst_diaSource_band')
                    snr = get_prop(alert, 'lsst_diaSource_snr')
                    scienceFlux = get_prop(alert, 'lsst_diaSource_scienceFlux')
                    psfFlux = get_prop(alert, 'lsst_diaSource_psfFlux')
                    apFlux = get_prop(alert, 'lsst_diaSource_apFlux')
                    extendedness = get_prop(alert, 'lsst_diaSource_extendedness')
                    psfChi2 = get_prop(alert, 'lsst_diaSource_psfChi2')
                    dipoleFitAttempted = get_prop(alert, 'lsst_diaSource_dipoleFitAttempted')
                    dipoleChi2 = get_prop(alert, 'lsst_diaSource_dipoleChi2')
                    dipoleLength = get_prop(alert, 'lsst_diaSource_dipoleLength')
    
                    x = get_prop(alert, 'lsst_diaSource_x')
                    y = get_prop(alert, 'lsst_diaSource_y')
                    ra = get_prop(alert, 'lsst_diaSource_ra')
                    dec = get_prop(alert, 'lsst_diaSource_dec')
                    engineered = _add_engineered_features(alert)
                    
                    feature_dict = {
                        # 'lsst_dia_obj_id': obj_id,
                        'locus_id': locus_id,
                        'alert_id': alert_num,
                        'n_alerts': n_alerts,
                        'ra': ra,
                        'dec': dec,
                        'x': x,
                        'y': y,
                        'band': band,
                        'psf_fwhm': engineered['psf_fwhm'],
                        'snr': snr,
                        'template_flux': engineered['template_flux'],
                        'scienceFlux': scienceFlux,
                        'psfFlux': psfFlux,
                        'apFlux': apFlux,
                        'temp_sci_flux_ratio': engineered['temp_sci_flux_ratio'],
                        'moment_ext': engineered['moment_ext'],
                        'ellip_ext': engineered['ellip_ext'],
                        'flux_ext': engineered['flux_ext'],
                        'extendedness': extendedness,
                        'psfChi2': psfChi2,
                        'dipoleFitAttempted': dipoleFitAttempted,
                        'dipoleChi2': dipoleChi2,
                        'dipoleLength': dipoleLength,
                        'x_y_err': engineered['x_y_err'],
                    }
    
                    ref = f'{locus_id}_alert_{alert_num}'
                    res[ref] = feature_dict
                    alert_num += 1
        except:
            print(f'Error at ObjID: {locus_id}')
            continue

In [ ]:
df = pd.DataFrame.from_dict(res, orient='index')

df = df.astype({'locus_id': str, #'lsst_dia_obj_id': float, 
                'ra': float, 
                'dec': float,
                'x': float, 
                'y': float, 
                'band': str,
                'psf_fwhm': float, 
                'snr': float,
                'template_flux': float, 
                'scienceFlux': float,
                'psfFlux': float, 
                'apFlux': float,
                'temp_sci_flux_ratio': float, 
                'moment_ext': float,
                'ellip_ext': float,
                'flux_ext': float,
                'extendedness': float,
                'psfChi2': float, 
                'dipoleFitAttempted': bool,
                'dipoleChi2': float, 
                'dipoleLength': float,
                'x_y_err': float, 
                'alert_id': int,
                'n_alerts': int,
               })

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np

# Create SkyCoord object from your coordinates
# Assuming you have 'ra' and 'dec' columns in degrees
coords = SkyCoord(ra=df[df.alert_id==0]['ra'].values*u.deg, 
                  dec=df[df.alert_id==0]['dec'].values*u.deg, 
                  frame='icrs')

# Match catalog to itself
idx, d2d, d3d = coords.match_to_catalog_sky(coords, nthneighbor=2)
# nthbest=2 gives the second-closest match (closest would be itself)

# Find matches within 3 arcsec
threshold = 3 * u.arcsec
mask = d2d < threshold

# Get pairs of matching sources
matched_indices = np.where(mask)[0]
matched_pairs = list(zip(matched_indices, idx[mask]))

print(f"Found {len(matched_indices)} sources with neighbors within 3 arcsec")

We currently **don't implement any clustering**, because when looking at 10000 Target loci, none of them have neighbours within that 3 arcsec radius. As we increase the number of Targets, we may need to change this. 

In [ ]:
# # Add results to dataframe if needed
# df['has_close_neighbor'] = mask
# df['nearest_neighbor_idx'] = idx
# df['nearest_neighbor_sep_arcsec'] = d2d.arcsec

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
summary_df = calculate_stats(df)
summary_df['n_lsst_alerts'] +=1  #adding 1 since we were indexing from zero

In [ ]:
n_tagged = []
ratio_tagged = []

for key, row in summary_df.iterrows():
    n_tags = row['n_tagged_alerts']
    n_alerts = row['n_detections']

    ratio = n_tags / n_alerts * 100
    
    n_tagged.append(n_tags)
    ratio_tagged.append(np.round(ratio, decimals=5))

summary_df = summary_df.drop(columns=['n_tagged_alerts', 'n_detections'])
summary_df['n_tagged'] = n_tagged
summary_df['percent_tagged'] = ratio_tagged

In [ ]:
X = summary_df.drop(columns=['locus_id', 'ra', 'dec', 'n_lsst_alerts']) #'category'
y_pred = lantern.predict(X)
y_prob = lantern.predict_proba(X)[:, 1]

In [ ]:
XGB_THRESHOLD = 0.8 #alter depending on rigidly you want to reject candidates
candidates = summary_df[y_prob > XGB_THRESHOLD]

In [ ]:
candidate_ids = candidates.locus_id.astype(str)
print(f'# of candidates from {lim} Target loci: {len(candidate_ids)}')

### Save the output Candidate IDs here!
An updated version of this should find some way of tracking _which_ Targets were searched in the query, since this code will not return the same Targets if ANTARES adds more tagged loci. 

In [ ]:
len(candidate_ids)

In [ ]:
import csv 

fileName = f'candidateID_secondPass_{XGB_THRESHOLD}XGB_from{lim}targets.csv'

with open(fileName, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(candidate_ids)

### And visualise!

In [ ]:
for locus in candidate_ids:
    try:
        show_by_id(locus)
    except Exception as e:
        print(f'{locus}: {e}')